# Stylized Painting with Gradient Descent

This notebook uses a trained neural network (that mimics some brush strokes) to optimize brush stroke parameters via gradient descent, recreating target images (optionally with style transfer).

## Quick Start

The notebook can be run as is without changing anything. It would paint sunflowers with oil-brush strokes and then apply style transfer to them using Picasso's painting.

You can however easily:
1. **Change brush style**: In the General Configuration cell, set `brush_type` parameter of `create_default_brush` function call to one of the following values `"watercolor"`, `"rectangle"` or `"texture"` (default). The notebook handles everything else automatically, including downloading pretrained weights from Hugging Face Hub.

2. **Provide your own images**: Fill `target_images` that you want to paint, or `style_transfer_pairs` for style transfer. Supported formats are: URL or local path.

If you wish for more customization, such as providing your own implementation of the `Brush` class, jump to the *Customization* section for more details.

#### Description of available brush styles
- **Watercolor** - curved strokes with bleeding edges that mimic watercolor paint spreading on wet paper.
- **Texture** - painterly oil-brush strokes with smooth color gradients from start to end.
- **Rectangle** - axis-aligned rectangular strokes with a subtle grainy watercolor texture.

## Algorithm Overview

The painting algorithm uses **hierarchical grid optimization with active sets**:

1. **Progressive Grid Splitting**: Canvas and target image are split into increasingly fine grids (e.g., 1×1 → 2×2 → ... → 5×5). Coarse grids produce large strokes for global structure; fine grids produce small strokes for details.

2. **Active Set Optimization**: For each grid, brush strokes are added incrementally to an "active set" and jointly optimized via gradient descent with specified image loss. Note that brush strokes belonging to neighboring grids don't interact with each other at all during optimization. This allows strokes to work together while keeping optimization tractable.

3. **Two-Brush System**: A differentiable neural network brush is used during optimization (for gradients), then the real non-differentiable brush renders the final strokes.

4. **Target-Guided Sampling**: Initial stroke parameters are sampled based on error between current canvas and target, focusing strokes where they're needed most.

5. **Rescaling**: After separate grid optimization, stroke parameters are rescaled to full canvas coordinates.

## Notebook Structure and Customization

This notebook is organized into three sections, each offering its own customizations:

1. **General Configuration** - Set up the basics: device (CPU/GPU), output canvas size, video generation flag, batch sizes, brush, neural renderer, sampler, loss and painter. Defaults work well for most use cases.

2. **Painting** - Specify your `results_folder` and define images to paint via `target_images` list. Both local file paths and URLs are supported.

3. **Style Transfer** - Specify your `style_transfer_pairs`: a list of `(target_image, [style_images])` tuples. Each target is painted once, then style-transferred with every style image in its list. You can also configure style transferer settings such as: color only mode, number of optimization steps or custom `style_loss`.

Check out individual cells for more details.

## Memory Management

Three batch size parameters control the memory/speed tradeoff:
- `grid_batch_size` (default: 50) - how many grids are optimized simultaneously during painting
- `rendering_batch_size` (default: 35) - how many brush strokes are rendered at once when drawing on the final canvas
- `style_transfer_rendering_batch_size` (default: 50) - batch size for rendering during style transfer optimization

If you encounter **out-of-memory (OOM) errors**, lower these values or lower `out_canvas_size`. The defaults work comfortably with 16 GB GPU memory (Kaggle/Colab free tier).

## Outputs

### Painting
Results are saved to `{results_folder}/results_{i}/` for each target image:
- `painted_canvas.png` -> painted canvas
- `painting_progress.mp4` -> painting video (if `should_generate_video` flag is set to True)
- `optimized_brush_params.pt` -> tensor brush parameters

Existing folders are overwritten.

### Style Transfer
Results are saved to `{style_results_folder}/results_{i}/` for each (target, style) pair:
- `painted_canvas.png` -> painted canvas (before style transfer)
- `stylized_painted_canvas.png` -> stylized painted canvas
- `optimized_brush_params.pt` -> tensor brush parameters (before style transfer)
- `stylized_optimized_brush_params.pt` -> stylized tensor brush parameters
- `stylized_painting_progress.mp4` -> stylized painting video

Existing folders are overwritten.

## Prerequisites

Trained `DifferentiableBrush` weights for the `Brush` you're painting with. The `train_nn_renderer.ipynb` notebook provides a customizable training loop for this.

## Implementation Details

The **Painter** class optimizes brush stroke parameters via gradient descent using a `DifferentiableBrush` (neural network). Its `paint()` method returns both the painted canvas and the optimized brush parameters, which can be saved for later rendering at arbitrary canvas sizes or used as input for style transfer.

The **StyleTransferer** class applies neural style transfer to existing brush parameters from `Painter.paint()`. It basically uses the same optimization algorithm as `Painter` but with a combined pixel and Gatys style loss, no initial random sampling (parameters are taken directly from the painter's output) and no progressively increasing number of smaller and smaller grids that are being optimized. Its `transfer()` method optimizes stroke parameters to match a style image while preserving content, returning stylized brush parameters that can be rendered with the real brush at arbitrary canvas sizes.

In [ ]:
import io
import warnings
from collections.abc import Iterable
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Literal
from urllib.parse import urlparse

import requests
import torch
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

from PIL import Image
from torch import Tensor
from torchvision.utils import save_image
from tqdm.auto import tqdm

from huggingface_hub import hf_hub_download

from painting.brushes import (
    Brush,
    DifferentiableBrush,
    GeneralNeuralRenderer,
    RectangleBrush,
    TextureBrush,
    WatercolorBrush,
    ColorMaskNeuralRenderer,
)
from painting.loss import PixelLoss
from painting.painter import Painter
from painting.samplers import TargetGuidedSampler
from painting.style_transferer import StyleTransferer
from painting.video_export import create_painting_video

In [ ]:
def _load_image_from_source(source: str) -> Image.Image:
    """Load PIL Image from local path or URL."""
    parsed = urlparse(source)
    if parsed.scheme in ("http", "https"):
        response = requests.get(source, timeout=30)
        response.raise_for_status()
        return Image.open(io.BytesIO(response.content))

    return Image.open(source)


def load_texture(source: str, device: str):
    """Load texture as grayscale tensor [1, H, W] in [0, 1] range."""
    img = _load_image_from_source(source).convert("L")
    tensor = TF.pil_to_tensor(img).float() / 255.0
    return tensor.to(device)


def load_image(source: str, device: str):
    """Load image as tensor [3, H, W] in [0, 255] range."""
    img = _load_image_from_source(source).convert("RGB")
    tensor = TF.pil_to_tensor(img).float()
    return tensor.to(device)

In [ ]:
def visualize_canvas(
    canvas: list[Tensor] | tuple[Tensor] | Tensor,
    gap: float = 0.1,
    titles: list[str] | None = None,
):
    """
    Visualize a single RGB canvas [0-255] or multiple canvases side by side.
    This function automatically resizes tensors to the same height while maintaining
    their resolution ratio.
    """
    if not isinstance(canvas, (list, tuple)):
        canvas = [canvas]

    tensors = [c.cpu().clamp(0, 255).to(torch.uint8) for c in canvas]

    min_h = min(t.shape[1] for t in tensors)
    resized = []
    for t in tensors:
        if t.shape[1] != min_h:
            scale = min_h / t.shape[1]
            new_w = int(t.shape[2] * scale)
            t = TF.resize(t, [min_h, new_w])
        resized.append(t)

    images = [t.permute(1, 2, 0).numpy() for t in resized]
    widths = [img.shape[1] for img in images]
    total_w = sum(widths)

    fig, axes = plt.subplots(
        1, len(images),
        figsize=(12, 12 * min_h / total_w),
        gridspec_kw={"width_ratios": widths, "wspace": gap},
    )
    if len(images) == 1:
        axes = [axes]

    for i, (ax, img) in enumerate(zip(axes, images)):
        ax.imshow(img)
        ax.axis("off")
        if titles and i < len(titles):
            ax.set_title(titles[i])

    plt.show()

In [ ]:
def tqdm_wrapper(iterable: Iterable, desc: str):
    items = list(iterable)
    if len(items) == 1:
        return items
    return tqdm(items, desc=desc, leave=False)

In [ ]:
@dataclass
class BrushMetadata:
    """Metadata about brush parameter indices."""
    params_count: int
    pos_indices: list[tuple[int, int]]  # Each position is (x, y) pair
    size_indices: list[int]
    color_indices: list[tuple[int, int, int]]  # Each color is (R, G, B) triplet


def create_default_brush(
    brush_type: Literal["watercolor", "texture", "rectangle"],
    canvas_size: int,
    device: str,
) -> tuple[Brush, BrushMetadata]:
    """Create brush with default configuration."""
    if brush_type == "watercolor":
        brush = WatercolorBrush(
            canvas_size=canvas_size,
            # We're using closing for better effect on bigger canvases.
            apply_closing=True,
            device=device,
        )
        metadata = BrushMetadata(
            params_count=brush.brush_params_count,
            pos_indices=[(0, 1), (2, 3), (4, 5)],
            size_indices=[6],
            color_indices=[(7, 8, 9)],
        )
        return brush, metadata
    elif brush_type == "texture":
        brush = TextureBrush(
            large_vertical=load_texture("./assets/textures/large_vertical_brush.png", device),
            large_horizontal=load_texture("./assets/textures/large_horizontal_brush.png", device),
            small_vertical=load_texture("./assets/textures/small_vertical_brush.png", device),
            small_horizontal=load_texture("./assets/textures/small_horizontal_brush.png", device),
            canvas_size=canvas_size,
            # We're using closing for better effect on bigger canvases.
            apply_closing=True,
            device=device,
        )
        metadata = BrushMetadata(
            params_count=brush.brush_params_count,
            pos_indices=[(0, 1)],
            size_indices=[2, 3],
            color_indices=[(5, 6, 7), (8, 9, 10)],
        )
        return brush, metadata
    elif brush_type == "rectangle":
        brush = RectangleBrush(
            canvas_size=canvas_size,
            device=device,
        )
        metadata = BrushMetadata(
            params_count=brush.brush_params_count,
            pos_indices=[(0, 1)],
            size_indices=[2, 3],
            color_indices=[(4, 5, 6)],
        )
        return brush, metadata
    else:
        raise ValueError(f"Unknown brush type: {brush_type}")


def create_pretrained_differentiable_brush(
    brush: Brush,
    device: str,
) -> DifferentiableBrush:
    """
    Create differentiable brush with default configuration for the given brush and loads
    weights from Hugging Face Hub.
    """
    brush_config = {
        WatercolorBrush: "Druudik1/watercolor-brush",
        TextureBrush: "Druudik1/texture-brush",
        RectangleBrush: "Druudik1/rectangle-brush",
    }
    brush_type = type(brush)
    if brush_type not in brush_config:
        raise ValueError(f"Unknown brush: {brush_type}")

    repo_id = brush_config[brush_type]

    # Note that canvas size of the differentiable brush doesn't need to match out_canvas_size,
    # because it's just used during brush params optimization, not for the final painting.
    if isinstance(brush, WatercolorBrush):
        differentiable_brush = ColorMaskNeuralRenderer(
            brush_params_count=brush.brush_params_count,
            color_params_indices=[7, 8, 9],
            canvas_size=128,
            device=device,
        )
    else:
        differentiable_brush = GeneralNeuralRenderer(
            brush_params_count=brush.brush_params_count,
            canvas_size=128,
            device=device,
        )

    weights = hf_hub_download(repo_id=repo_id, filename="neural_renderer.pt")
    differentiable_brush.load_state_dict(
        torch.load(weights, map_location=device, weights_only=True)
    )
    return differentiable_brush

# General Configuration

Define device, canvas size, brushes, and painter settings. Defaults use TextureBrush with GeneralNeuralRenderer.

**Tip:** For custom brushes, implement the `Brush` and `DifferentiableBrush` base classes and plug them in below.

In [ ]:
# === Device & Canvas ===
device = "cuda" if torch.cuda.is_available() else "cpu"

# Final canvas size on which optimized brush strokes will be drawn. Can be arbitrarily large.
out_canvas_size = 512

# Instead of using black initial canvas, one can use any other color or pattern.
# From my testing, black canvas seems to work quite good in general case.
# But it can be good idea to tweak it based on the image we're trying
# to paint (e.g. if the majority of the image is blue sky, we can just set background to blue
# which will make optimization a bit easier; but even black initial canvas can handle
# that easily with progressive painting).
initial_canvas = torch.zeros((3, out_canvas_size, out_canvas_size), device=device)

# Video generation takes ~15 seconds per painting (depending on the brush complexity and
# output canvas size).
should_generate_video = False

# === Brush & Neural Renderer ===

# Specify your own brush and its metadata or use default.
brush: Brush | None = None
brush_metadata: BrushMetadata | None = None

# Both must be defined or both must be undefined and default will be used.
assert (brush is None) == (brush_metadata is None)

if brush is None:
    # You can easily change brush_type to either on of these values and the notebook will run
    # without any changes: texture, watercolor, rectangle
    brush, brush_metadata = create_default_brush(
        brush_type="texture",
        canvas_size=out_canvas_size,
        device=device,
    )

# Define differentiable brush (with its weights checkpoint) or use default.
# This differentiable brush should mimic brush defined above.
# It will be used during optimization of the brush strokes, not during final painting.
differentiable_brush: DifferentiableBrush | None = None
differentiable_brush_checkpoint: str | None = None

# Both must be defined or both must be undefined and default will be used.
assert (differentiable_brush is None) == (differentiable_brush_checkpoint is None)

if differentiable_brush is None:
    differentiable_brush = create_pretrained_differentiable_brush(
        brush=brush,
        device=device,
    )
else:
    differentiable_brush.load_state_dict(
        torch.load(differentiable_brush_checkpoint, map_location=device, weights_only=True)
    )

differentiable_brush.eval()

# === Sampler, Loss & Painter ===

# This parameter represents how many grids can be optimized simultaneously.
# In a case of OOM errors, one can lower it to match the memory capacity of the GPU that's used.
grid_batch_size = 50

# This parameter represents how many brush strokes are rendered at once before painting.
# In a case of OOM errors, one can lower it to match the memory capacity of the GPU that's used.
rendering_batch_size = 35

# This parameter represents how many brush strokes are rendered at once using differentiable
# brush during style transfer optimization.
# In a case of OOM errors, one can lower it to match the memory capacity of the GPU that's used.
style_transfer_rendering_batch_size = 50

# Position parameters will be constrained to [boundary_offset, 1-boundary_offset] during optimization.
# This is **IMPORTANT** to not have strokes reaching neighboring grids on the final canvas (which
# they didn't see during optimization).
boundary_offset = 0.1

# Size parameters will be constrained to [min_local_param_size, max_local_param_size]
# during optimization. This is **IMPORTANT** since it prevents gradient descent from
# making strokes too small (poor gradient signal) or too large (strokes may extend
# outside their grid; this works in combination with boundary_offset).
min_local_param_size = 0.1
max_local_param_size = 0.9

sampler = TargetGuidedSampler(
    brush_params_count=brush.brush_params_count,
    pos_param_indices=brush_metadata.pos_indices,
    size_param_indices=brush_metadata.size_indices,
    color_param_indices=brush_metadata.color_indices,
    boundary_offset=boundary_offset,
    device=device,
)

loss = PixelLoss(p=1)

painter = Painter(
    brush=brush,
    brush_pos_indices=brush_metadata.pos_indices,
    brush_size_indices=brush_metadata.size_indices,
    differentiable_brush_imitator=differentiable_brush,
    brush_param_sampler=sampler,
    loss=loss,

    boundary_offset=boundary_offset,
    min_local_param_size=min_local_param_size,
    max_local_param_size=max_local_param_size,

    grid_batch_size=grid_batch_size,
    rendering_batch_size=rendering_batch_size,

    device=device,
)

In [ ]:
# Check consistency between boundary_offset parameters.
if painter.boundary_offset != sampler.boundary_offset:
    warnings.warn(
        f"boundary_offset differs: painter={painter.boundary_offset}, "
        f"sampler={sampler.boundary_offset} (this may be intentional)"
    )

# Painting

Specify target images and optionally override painting schedule parameters.

- `target_images`: List of image sources (local paths or URLs)
- `painting_schedule_kwargs`: Optional per-image schedule overrides (see `painter.paint()`
   for available parameters). **Different brushes and paintings might require different
   schedule for the best outcome.**

With the default params it takes ~2 minutes to paint the image and around ~15 seconds
to generate video (measured on the Kaggle with free GPU P100).

In [ ]:
# Folder where painting results will be saved. Output consists of painted png, optimized brush params
# and optionally an mp4 painting video.
results_folder = "painting_results"

# Supported formats: HTTP URL or local file path.
# There are some images in the repository that you can play with right away.
target_images = [
    "./assets/images/sunflowers.jpg",
]

# In the paper, they were using dilation on the foreground and erosion
# on the alpha mask (I call this operation closing) for every rendered stroke
# from the neural network during optimization. This seems to help in certain
# cases with optimization, but I've found out that it doesn't necessarily have
# positive impact for painting. However, when we're doing style transfer as well,
# using "closing" helps to produce better styled images in some cases. One can play
# with this flag and see which results he likes the best.
apply_closing_during_optim_in_painter = False

# Optional per-image schedule overrides (or None/empty dict for defaults)
# Check painter.paint() method for available schedule parameters overrides.
# Just note that the list parameters must be of the same size.
painting_schedule_kwargs: list[dict] | None = None

if painting_schedule_kwargs is None:
    if isinstance(brush, RectangleBrush):
        default_schedule = {
            "n_grids_per_dim_schedule": [1, 2, 4],
            "n_strokes_per_grid_schedule": [50, 5, 2],
            "active_set_size_schedule": [25, 5, 2],
            "total_optim_steps_per_active_set_schedule": [650, 250, 100],
        }
    elif isinstance(brush, WatercolorBrush):
        default_schedule = {
            "n_grids_per_dim_schedule": [1, 4, 5],
            "n_strokes_per_grid_schedule": [25, 5, 25],
            "active_set_size_schedule": [25, 5, 25],
            "total_optim_steps_per_active_set_schedule": [400, 200, 400],
        }
    else:
        # Use whatever default is set up in the painter class.
        default_schedule = {}

    painting_schedule_kwargs = [default_schedule for _ in range(len(target_images))]

assert len(target_images) == len(painting_schedule_kwargs)

**Run painting logic with the above configuration.**

In [ ]:
def paint_image(target: str, schedule_kwargs: dict, folder_to_save: Path) -> tuple[Tensor, Tensor]:
    """Paint a single image and save results."""
    if folder_to_save.exists():
        warnings.warn(
            f"Folder {folder_to_save} already exists. "
            f"Its content will be overriden."
        )

    folder_to_save.mkdir(parents=True, exist_ok=True)

    target_img = load_image(target, device)

    canvas, brush_params = painter.paint(
        target_image=target_img,
        initial_canvas=initial_canvas,
        apply_brush_stroke_closing_during_optim=apply_closing_during_optim_in_painter,
        iter_progress_wrapper=tqdm_wrapper,
        **schedule_kwargs,
    )

    save_image((canvas / 255).detach().cpu(), folder_to_save / "painted_canvas.png")
    torch.save(brush_params.detach().cpu(), folder_to_save / "optimized_brush_params.pt")

    print(f"[{datetime.now()}] Finished painting.\n")

    if should_generate_video:
        print(f"[{datetime.now()}] Starting painting video generation...\n")
        # One can specify other parameters affecting video speed, layout etc. to customize it.
        create_painting_video(
            brush=brush,
            brush_params=brush_params,
            initial_canvas=initial_canvas,
            output_path=folder_to_save / "painting_progress.mp4",
            target_images=target_img,
            target_layout="corner",
            brush_stroke_rendering_batch_size=rendering_batch_size,
            speed_schedule=[7, 15, 20, 60, 120],
            speed_milestones=[20, 100, 280, 600, 1100],
            fps=20,
        )
        print(f"[{datetime.now()}] Finished video generation.\n")

    visualize_canvas([canvas, target_img], titles=["Painted", "Target Image"])
    return canvas, brush_params


for i, (target, schedule_kwargs) in enumerate(zip(target_images, painting_schedule_kwargs)):
    print(f"[{datetime.now()}] Painting image {i+1}/{len(target_images)}: {target}")
    paint_image(target, schedule_kwargs, Path(results_folder) / f"results_{i}")

# Style Transfer

Apply neural style transfer to brush stroke parameters. First paints the content image, then transfers style.

**Important:** Style transfer requires `n_grids_per_dim_schedule` to have a single value (all brush strokes must be optimized at same grid resolution).

The default style transfer setup uses fewer brush strokes, fewer optimization steps and doesn't use progressive painting.
It might not create the most polished and realistic paintings, but it works good enough for style transfer.

Each target image is painted **once**, then style-transferred with every style image in its list.
This avoids redundant repainting when applying multiple styles to the same target.
Configure pairs via `style_transfer_pairs: list[tuple[str, list[str]]]`.

For **custom style loss**, implement the `ImageLoss` base class and pass it to `StyleTransferer`.
You can also play with existing `GatysStyleLoss` and try different param combinations.

Performance note: with the default params it takes ~40 seconds to paint the image and then each
subsequent style transfer of that image takes around ~1 minute (measured on the Kaggle with free GPU P100).

In [ ]:
# === Style Transfer Settings ===
style_results_folder = "style_painting_results"

# For the description, check apply_brush_stroke_closing_during_optim_in_painter flag
# in the Painting section.
apply_closing_during_optim_style_transfer = True

# Set optimization style transfer parameters.
# Style transfer first optimizes brush parameters against the target image, then applies
# style transformation. Schedule parameters control the initial painting phase.
# Flag color_only_mode controls whether only color params will be optimized during style transfer.
# Note: Style transfer uses a fixed grid size. Using multiple values in n_grids_per_dim_schedule
# will cause strokes to be truncated at grid edges when resolution changes between iterations.
# For best results, use a single value matching n_grids_per_dim_style_transfer.
if isinstance(brush, RectangleBrush):
    color_only_mode = True
    n_grids_per_dim_schedule = [1]
    n_strokes_per_grid_schedule = [125]
    active_set_size_schedule = [25]
    total_optim_steps_per_active_set_schedule = [350]
elif isinstance(brush, WatercolorBrush):
    color_only_mode = False
    n_grids_per_dim_schedule = [5]
    n_strokes_per_grid_schedule = [25]
    active_set_size_schedule = [25]
    total_optim_steps_per_active_set_schedule = [400]
else:
    color_only_mode = False
    n_grids_per_dim_schedule = [5]
    n_strokes_per_grid_schedule = [40]
    active_set_size_schedule = [20]
    total_optim_steps_per_active_set_schedule = [240]

# Grid divisions for style transfer. Each stroke is assigned to one grid cell based on
# position and optimized only against that cell. Position/size clipping also respects
# grid boundaries.
style_transfer_n_grids_per_dim = n_grids_per_dim_schedule[0]

# Style transferer (uses differentiable_brush from general configuration cell).
style_transferer = StyleTransferer(
    differentiable_brush=differentiable_brush,
    brush_pos_indices=brush_metadata.pos_indices,
    brush_size_indices=brush_metadata.size_indices,
    brush_color_indices=brush_metadata.color_indices,
    color_only_mode=color_only_mode,
    boundary_offset=boundary_offset,
    min_local_param_size=min_local_param_size,
    max_local_param_size=max_local_param_size,
    rendering_batch_size=style_transfer_rendering_batch_size,
    device=device,

    # One can optionally also define its own pixel and style weights with their weights to
    # produce vastly different style effects. Check out GatysStyleLoss for more info. Putting different
    # weights on different layers can produce very different results.
    # pixel_loss=None,
    # style_loss=None,
)

# Specify target images paired with their style images.
# Each entry is (target_image_source, [list_of_style_image_sources]).
# The target is painted once, then style-transferred with each style image.
# Supported formats: HTTP URL or local file path.
style_transfer_pairs: list[tuple[str, list[str]]] = [
    ("./assets/images/sunflowers.jpg", ["./assets/images/picasso.jpg"]),
]

**Run style transfer logic with the above configuration.**

In [ ]:
def paint_and_style_transfer_image(
    target_img_source: str,
    style_img_sources: list[str],
    target_folder: Path,
):
    """Paint content image once, then apply style transfer for each style image."""
    if target_folder.exists():
        warnings.warn(
            f"Folder {target_folder} already exists. "
            f"Its content will be overriden."
        )
    target_folder.mkdir(parents=True, exist_ok=True)

    target_img = load_image(target_img_source, device)

    # Paint content image once
    print(f"[{datetime.now()}] Optimizing brush params for style transfer...\n")
    canvas, brush_params = painter.paint(
        target_image=target_img,
        initial_canvas=initial_canvas,
        n_grids_per_dim_schedule=n_grids_per_dim_schedule,
        n_strokes_per_grid_schedule=n_strokes_per_grid_schedule,
        active_set_size_schedule=active_set_size_schedule,
        total_optim_steps_per_active_set_schedule=total_optim_steps_per_active_set_schedule,
        apply_brush_stroke_closing_during_optim=apply_closing_during_optim_style_transfer,
        iter_progress_wrapper=tqdm_wrapper,
    )

    # Save painting results once per target
    save_image((canvas / 255).detach().cpu(), target_folder / "painted_canvas.png")
    torch.save(brush_params.detach().cpu(), target_folder / "optimized_brush_params.pt")

    for style_idx, style_img_source in enumerate(style_img_sources):
        style_folder = target_folder / f"style_{style_idx}"
        style_folder.mkdir(parents=True, exist_ok=True)

        style_img = load_image(style_img_source, device)

        # Apply style transfer
        print(f"[{datetime.now()}] Starting style transfer {style_idx}: {style_img_source}...\n")
        stylized_brush_params = style_transferer.transfer(
            brush_params=brush_params,
            target_image=target_img,
            style_image=style_img,
            n_grids_per_dim=style_transfer_n_grids_per_dim,
            initial_canvas=initial_canvas,
            apply_brush_stroke_closing=apply_closing_during_optim_style_transfer,
            iter_progress_wrapper=tqdm_wrapper,
        )

        stylized_canvas = brush.draw_on_single_canvas(
            brush_params=stylized_brush_params,
            canvas=initial_canvas,
            rendering_batch_size=rendering_batch_size,
        )

        # Save stylized painted canvas and brush params
        save_image((stylized_canvas / 255).detach().cpu(), style_folder / "stylized_painted_canvas.png")
        torch.save(stylized_brush_params.detach().cpu(), style_folder / "stylized_optimized_brush_params.pt")

        print(f"[{datetime.now()}] Finished style transfer.\n")

        if should_generate_video:
            print(f"[{datetime.now()}] Starting painting video generation...\n")
            create_painting_video(
                brush=brush,
                brush_params=stylized_brush_params,
                initial_canvas=initial_canvas,
                output_path=style_folder / "stylized_painting_progress.mp4",
                target_images=[target_img, style_img],
                target_layout="corner",
                brush_stroke_rendering_batch_size=rendering_batch_size,
                speed_schedule=[30],
                speed_milestones=[500],
                fps=30,
            )
            print(f"[{datetime.now()}] Finished video generation.\n")

        visualize_canvas(
            canvas=[target_img, style_img],
            titles=["Target Image", "Style Image"],
        )
        visualize_canvas(
            canvas=[canvas, stylized_canvas],
            titles=["Painted", "Stylized"],
        )


for target_idx, (target_img_source, style_img_sources) in enumerate(style_transfer_pairs):
    print(f"[{datetime.now()}] Painting target {target_idx}: {target_img_source}")
    target_folder = Path(style_results_folder) / f"target_{target_idx}"
    paint_and_style_transfer_image(target_img_source, style_img_sources, target_folder)